# Bank Churners — Exploratory Data Analysis

**Goal:** Understand the dataset, assess data quality, and identify associations with customer churn.

**Notes:**
- Correlation and visual patterns describe **associations**, not proven causes.
- Two Naive Bayes columns are **leakage** and must be excluded from modeling.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# Colorblind-friendly palette (Okabe-Ito inspired)
CHURN_PALETTE = {0: "#0072B2", 1: "#D55E00"}
CHURN_LABELS = {0: "Existing", 1: "Attrited"}

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "BankChurners.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "BankChurners.csv"
FIGURES_DIR = ROOT / "reports" / "figures"
TABLES_DIR = ROOT / "reports" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0


def save_fig(name: str) -> Path:
    """Save current figure to reports/figures with stable naming."""
    global _fig_counter
    _fig_counter += 1
    path = FIGURES_DIR / f"eda_{_fig_counter:02d}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path.relative_to(ROOT)}")
    return path


## 1. Data Loading

In [2]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
display(df_raw.head())
display(df_raw.dtypes.to_frame("dtype"))


Shape: 10,127 rows × 23 columns


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


,dtype
CLIENTNUM,int64
Attrition_Flag,object
Customer_Age,int64
Gender,object
Dependent_count,int64
Education_Level,object
Marital_Status,object
Income_Category,object
Card_Category,object
Months_on_book,int64


In [3]:
LEAKAGE_COLS = [
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2",
]
ID_COLS = ["CLIENTNUM"]
TARGET_COL = "Attrition_Flag"

df = df_raw.copy()
df["is_churn"] = (df[TARGET_COL] == "Attrited Customer").astype(int)

numeric_cols = [
    "Customer_Age", "Dependent_count", "Months_on_book", "Total_Relationship_Count",
    "Months_Inactive_12_mon", "Contacts_Count_12_mon", "Credit_Limit",
    "Total_Revolving_Bal", "Avg_Open_To_Buy", "Total_Amt_Chng_Q4_Q1",
    "Total_Trans_Amt", "Total_Trans_Ct", "Total_Ct_Chng_Q4_Q1", "Avg_Utilization_Ratio",
]
categorical_cols = [
    "Gender", "Education_Level", "Marital_Status", "Income_Category", "Card_Category",
]
exclude_from_modeling = ID_COLS + LEAKAGE_COLS + [TARGET_COL]

print("Columns excluded from modeling:", exclude_from_modeling)
print(f"Modeling feature count (excl. target): {len(numeric_cols) + len(categorical_cols)}")


Columns excluded from modeling: ['CLIENTNUM', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2', 'Attrition_Flag']
Modeling feature count (excl. target): 19


**Loading summary**
- Target: `Attrition_Flag` → binary `is_churn` (1 = Attrited Customer).
- **Do not use for modeling:** `CLIENTNUM` (identifier) and both **Naive Bayes classifier** columns (precomputed scores → leakage).


## 2. Data Quality

In [4]:
missing = df_raw.isna().sum().sort_values(ascending=False)
unknown_counts = {}
for col in categorical_cols:
    unknown_counts[col] = int((df[col] == "Unknown").sum())

quality_summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": missing,
    "missing_pct": (missing / len(df_raw) * 100).round(2),
})
for col, cnt in unknown_counts.items():
    quality_summary.loc[col, "unknown_literal_count"] = cnt

quality_summary["duplicate_rows"] = df_raw.duplicated().sum()
quality_summary.loc["CLIENTNUM", "duplicate_ids"] = df_raw["CLIENTNUM"].duplicated().sum()

quality_summary.to_csv(TABLES_DIR / "eda_data_quality_summary.csv")
display(quality_summary.head(15))
print("No NaN values in raw file." if missing.sum() == 0 else "Missing values present.")


,dtype,missing_count,missing_pct,unknown_literal_count,duplicate_rows,duplicate_ids
CLIENTNUM,int64,0,0.0,NaN,0,0.0
Attrition_Flag,object,0,0.0,NaN,0,NaN
Customer_Age,int64,0,0.0,NaN,0,NaN
Gender,object,0,0.0,0.0,0,NaN
Dependent_count,int64,0,0.0,NaN,0,NaN
Education_Level,object,0,0.0,1519.0,0,NaN
Marital_Status,object,0,0.0,749.0,0,NaN
Income_Category,object,0,0.0,1112.0,0,NaN
Card_Category,object,0,0.0,0.0,0,NaN
Months_on_book,int64,0,0.0,NaN,0,NaN


No NaN values in raw file.


**Data quality (initial)**
- No null values in the CSV; watch for literal **"Unknown"** in categoricals (`Education_Level`, `Marital_Status`, `Income_Category`).
- No duplicate rows or duplicate `CLIENTNUM` in this extract.


In [5]:
churn_counts = df["is_churn"].value_counts().sort_index()
churn_rate = df["is_churn"].mean()
print(churn_counts.rename(index=CHURN_LABELS))
print(f"Churn rate: {churn_rate:.1%} ({churn_counts.get(1, 0):,} attrited / {len(df):,} customers)")

fig, ax = plt.subplots(figsize=(7, 4))
bars = sns.barplot(
    x=churn_counts.index.map(CHURN_LABELS),
    y=churn_counts.values,
    hue=churn_counts.index.map(CHURN_LABELS),
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False,
    ax=ax,
)
ax.set_title("Target distribution: customer attrition")
ax.set_xlabel("Customer status")
ax.set_ylabel("Count")
for container in bars.containers:
    for bar in container:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h, f"{int(h):,}", ha="center", va="bottom")
save_fig("target_distribution_count")

fig, ax = plt.subplots(figsize=(6, 4))
props = churn_counts / churn_counts.sum()
ax.bar(props.index.map(CHURN_LABELS), props.values, color=[CHURN_PALETTE[0], CHURN_PALETTE[1]])
ax.set_title("Target distribution: churn proportion")
ax.set_ylabel("Proportion")
ax.set_ylim(0, 1)
for i, v in enumerate(props.values):
    ax.text(i, v + 0.02, f"{v:.1%}", ha="center")
save_fig("target_distribution_proportion")


is_churn
Existing    8500
Attrited    1627
Name: count, dtype: int64
Churn rate: 16.1% (1,627 attrited / 10,127 customers)


Saved: reports/figures/eda_01_target_distribution_count.png
Saved: reports/figures/eda_02_target_distribution_proportion.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_02_target_distribution_proportion.png')

**Class imbalance:** Roughly **16%** of customers are attrited. Retention campaigns should account for imbalance in any future classification model (e.g., stratified splits, appropriate metrics).


In [6]:
# Outlier screen (IQR-based flag counts)
outlier_flags = {}
for col in numeric_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_flags[col] = int(((df[col] < lower) | (df[col] > upper)).sum())

outlier_df = pd.Series(outlier_flags, name="iqr_outlier_count").sort_values(ascending=False)
display(outlier_df.to_frame())

util_high = (df["Avg_Utilization_Ratio"] > 1).sum()
print(f"Avg_Utilization_Ratio > 1: {util_high} rows (review if unexpected)")


,iqr_outlier_count
Credit_Limit,984
Avg_Open_To_Buy,963
Total_Trans_Amt,896
Contacts_Count_12_mon,629
Total_Amt_Chng_Q4_Q1,396
Total_Ct_Chng_Q4_Q1,394
Months_on_book,386
Months_Inactive_12_mon,331
Customer_Age,2
Total_Trans_Ct,2


Avg_Utilization_Ratio > 1: 0 rows (review if unexpected)


**Leakage reminder:** The two `Naive_Bayes_Classifier_*` fields are model outputs related to attrition. They are excluded from EDA correlation heatmaps for modeling features and must not enter training.


## 3. Univariate Analysis

In [7]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Customer_Age", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Customer_Age", bins=range(int(df["Customer_Age"].min()), int(df["Customer_Age"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: customer age")
ax.set_xlabel("Customer_Age")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_customer_age")


Saved: reports/figures/eda_03_univariate_customer_age.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_03_univariate_customer_age.png')

**Customer_Age:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [8]:
fig, ax = plt.subplots(figsize=(7, 4))
if "hist" == "kde":
    sns.kdeplot(data=df, x="Dependent_count", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Dependent_count", bins=range(int(df["Dependent_count"].min()), int(df["Dependent_count"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: dependent count")
ax.set_xlabel("Dependent_count")
ax.set_ylabel("Density" if "hist" == "kde" else "Count")
save_fig("univariate_dependent_count")


Saved: reports/figures/eda_04_univariate_dependent_count.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_04_univariate_dependent_count.png')

**Dependent_count:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [9]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Months_on_book", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Months_on_book", bins=range(int(df["Months_on_book"].min()), int(df["Months_on_book"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: months on book")
ax.set_xlabel("Months_on_book")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_months_on_book")


Saved: reports/figures/eda_05_univariate_months_on_book.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_05_univariate_months_on_book.png')

**Months_on_book:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [10]:
fig, ax = plt.subplots(figsize=(7, 4))
if "hist" == "kde":
    sns.kdeplot(data=df, x="Total_Relationship_Count", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Total_Relationship_Count", bins=range(int(df["Total_Relationship_Count"].min()), int(df["Total_Relationship_Count"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: total relationship count")
ax.set_xlabel("Total_Relationship_Count")
ax.set_ylabel("Density" if "hist" == "kde" else "Count")
save_fig("univariate_total_relationship_count")


Saved: reports/figures/eda_06_univariate_total_relationship_count.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_06_univariate_total_relationship_count.png')

**Total_Relationship_Count:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [11]:
fig, ax = plt.subplots(figsize=(7, 4))
if "hist" == "kde":
    sns.kdeplot(data=df, x="Months_Inactive_12_mon", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Months_Inactive_12_mon", bins=range(int(df["Months_Inactive_12_mon"].min()), int(df["Months_Inactive_12_mon"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: months inactive (12 mon)")
ax.set_xlabel("Months_Inactive_12_mon")
ax.set_ylabel("Density" if "hist" == "kde" else "Count")
save_fig("univariate_months_inactive_12_mon")


Saved: reports/figures/eda_07_univariate_months_inactive_12_mon.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_07_univariate_months_inactive_12_mon.png')

**Months_Inactive_12_mon:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [12]:
fig, ax = plt.subplots(figsize=(7, 4))
if "hist" == "kde":
    sns.kdeplot(data=df, x="Contacts_Count_12_mon", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Contacts_Count_12_mon", bins=range(int(df["Contacts_Count_12_mon"].min()), int(df["Contacts_Count_12_mon"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: contacts count (12 mon)")
ax.set_xlabel("Contacts_Count_12_mon")
ax.set_ylabel("Density" if "hist" == "kde" else "Count")
save_fig("univariate_contacts_count_12_mon")


Saved: reports/figures/eda_08_univariate_contacts_count_12_mon.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_08_univariate_contacts_count_12_mon.png')

**Contacts_Count_12_mon:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [13]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Credit_Limit", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Credit_Limit", bins=range(int(df["Credit_Limit"].min()), int(df["Credit_Limit"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: credit limit")
ax.set_xlabel("Credit_Limit")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_credit_limit")


Saved: reports/figures/eda_09_univariate_credit_limit.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_09_univariate_credit_limit.png')

**Credit_Limit:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [14]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Total_Revolving_Bal", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Total_Revolving_Bal", bins=range(int(df["Total_Revolving_Bal"].min()), int(df["Total_Revolving_Bal"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: total revolving balance")
ax.set_xlabel("Total_Revolving_Bal")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_total_revolving_bal")


Saved: reports/figures/eda_10_univariate_total_revolving_bal.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_10_univariate_total_revolving_bal.png')

**Total_Revolving_Bal:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [15]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Avg_Open_To_Buy", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Avg_Open_To_Buy", bins=range(int(df["Avg_Open_To_Buy"].min()), int(df["Avg_Open_To_Buy"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: avg open to buy")
ax.set_xlabel("Avg_Open_To_Buy")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_avg_open_to_buy")


Saved: reports/figures/eda_11_univariate_avg_open_to_buy.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_11_univariate_avg_open_to_buy.png')

**Avg_Open_To_Buy:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [16]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Total_Amt_Chng_Q4_Q1", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Total_Amt_Chng_Q4_Q1", bins=range(int(df["Total_Amt_Chng_Q4_Q1"].min()), int(df["Total_Amt_Chng_Q4_Q1"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: total amount change Q4 vs Q1")
ax.set_xlabel("Total_Amt_Chng_Q4_Q1")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_total_amt_chng_q4_q1")


Saved: reports/figures/eda_12_univariate_total_amt_chng_q4_q1.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_12_univariate_total_amt_chng_q4_q1.png')

**Total_Amt_Chng_Q4_Q1:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [17]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Total_Trans_Amt", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Total_Trans_Amt", bins=range(int(df["Total_Trans_Amt"].min()), int(df["Total_Trans_Amt"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: total transaction amount")
ax.set_xlabel("Total_Trans_Amt")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_total_trans_amt")


Saved: reports/figures/eda_13_univariate_total_trans_amt.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_13_univariate_total_trans_amt.png')

**Total_Trans_Amt:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [18]:
fig, ax = plt.subplots(figsize=(7, 4))
if "hist" == "kde":
    sns.kdeplot(data=df, x="Total_Trans_Ct", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Total_Trans_Ct", bins=range(int(df["Total_Trans_Ct"].min()), int(df["Total_Trans_Ct"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: total transaction count")
ax.set_xlabel("Total_Trans_Ct")
ax.set_ylabel("Density" if "hist" == "kde" else "Count")
save_fig("univariate_total_trans_ct")


Saved: reports/figures/eda_14_univariate_total_trans_ct.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_14_univariate_total_trans_ct.png')

**Total_Trans_Ct:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [19]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Total_Ct_Chng_Q4_Q1", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Total_Ct_Chng_Q4_Q1", bins=range(int(df["Total_Ct_Chng_Q4_Q1"].min()), int(df["Total_Ct_Chng_Q4_Q1"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: total count change Q4 vs Q1")
ax.set_xlabel("Total_Ct_Chng_Q4_Q1")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_total_ct_chng_q4_q1")


Saved: reports/figures/eda_15_univariate_total_ct_chng_q4_q1.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_15_univariate_total_ct_chng_q4_q1.png')

**Total_Ct_Chng_Q4_Q1:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [20]:
fig, ax = plt.subplots(figsize=(7, 4))
if "kde" == "kde":
    sns.kdeplot(data=df, x="Avg_Utilization_Ratio", fill=True, color="#0072B2", ax=ax)
else:
    sns.histplot(data=df, x="Avg_Utilization_Ratio", bins=range(int(df["Avg_Utilization_Ratio"].min()), int(df["Avg_Utilization_Ratio"].max()) + 2),
                 color="#0072B2", ax=ax)
ax.set_title("Univariate: avg utilization ratio")
ax.set_xlabel("Avg_Utilization_Ratio")
ax.set_ylabel("Density" if "kde" == "kde" else "Count")
save_fig("univariate_avg_utilization_ratio")


Saved: reports/figures/eda_16_univariate_avg_utilization_ratio.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_16_univariate_avg_utilization_ratio.png')

**Avg_Utilization_Ratio:** Distribution shape informs scaling and outlier handling in modeling; compare to bivariate section for churn differences.

In [21]:
order = df["Gender"].value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y="Gender", order=order, color="#56B4E9", ax=ax)
ax.set_title("Univariate: Gender (counts)")
ax.set_xlabel("Count")
save_fig("univariate_gender_count")

fig, ax = plt.subplots(figsize=(8, 4))
props = df["Gender"].value_counts(normalize=True).reindex(order)
props.plot(kind="barh", color="#56B4E9", ax=ax)
ax.set_title("Univariate: Gender (proportion)")
ax.set_xlabel("Proportion")
save_fig("univariate_gender_proportion")


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: reports/figures/eda_17_univariate_gender_count.png


Saved: reports/figures/eda_18_univariate_gender_proportion.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_18_univariate_gender_proportion.png')

In [22]:
order = df["Education_Level"].value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y="Education_Level", order=order, color="#56B4E9", ax=ax)
ax.set_title("Univariate: Education_Level (counts)")
ax.set_xlabel("Count")
save_fig("univariate_education_level_count")

fig, ax = plt.subplots(figsize=(8, 4))
props = df["Education_Level"].value_counts(normalize=True).reindex(order)
props.plot(kind="barh", color="#56B4E9", ax=ax)
ax.set_title("Univariate: Education_Level (proportion)")
ax.set_xlabel("Proportion")
save_fig("univariate_education_level_proportion")


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: reports/figures/eda_19_univariate_education_level_count.png
Saved: reports/figures/eda_20_univariate_education_level_proportion.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_20_univariate_education_level_proportion.png')

In [23]:
order = df["Marital_Status"].value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y="Marital_Status", order=order, color="#56B4E9", ax=ax)
ax.set_title("Univariate: Marital_Status (counts)")
ax.set_xlabel("Count")
save_fig("univariate_marital_status_count")

fig, ax = plt.subplots(figsize=(8, 4))
props = df["Marital_Status"].value_counts(normalize=True).reindex(order)
props.plot(kind="barh", color="#56B4E9", ax=ax)
ax.set_title("Univariate: Marital_Status (proportion)")
ax.set_xlabel("Proportion")
save_fig("univariate_marital_status_proportion")


/tmp/ipykernel_1282531/956149545.py:2: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(8, 4))


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: reports/figures/eda_21_univariate_marital_status_count.png
Saved: reports/figures/eda_22_univariate_marital_status_proportion.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_22_univariate_marital_status_proportion.png')

In [24]:
order = df["Income_Category"].value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y="Income_Category", order=order, color="#56B4E9", ax=ax)
ax.set_title("Univariate: Income_Category (counts)")
ax.set_xlabel("Count")
save_fig("univariate_income_category_count")

fig, ax = plt.subplots(figsize=(8, 4))
props = df["Income_Category"].value_counts(normalize=True).reindex(order)
props.plot(kind="barh", color="#56B4E9", ax=ax)
ax.set_title("Univariate: Income_Category (proportion)")
ax.set_xlabel("Proportion")
save_fig("univariate_income_category_proportion")


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: reports/figures/eda_23_univariate_income_category_count.png


Saved: reports/figures/eda_24_univariate_income_category_proportion.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_24_univariate_income_category_proportion.png')

In [25]:
order = df["Card_Category"].value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y="Card_Category", order=order, color="#56B4E9", ax=ax)
ax.set_title("Univariate: Card_Category (counts)")
ax.set_xlabel("Count")
save_fig("univariate_card_category_count")

fig, ax = plt.subplots(figsize=(8, 4))
props = df["Card_Category"].value_counts(normalize=True).reindex(order)
props.plot(kind="barh", color="#56B4E9", ax=ax)
ax.set_title("Univariate: Card_Category (proportion)")
ax.set_xlabel("Proportion")
save_fig("univariate_card_category_proportion")


Saved: reports/figures/eda_25_univariate_card_category_count.png
Saved: reports/figures/eda_26_univariate_card_category_proportion.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_26_univariate_card_category_proportion.png')

## 4. Bivariate Analysis (vs churn)

In [26]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Months_Inactive_12_mon", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Months_Inactive_12_mon by churn status")
ax.set_xlabel("Months_Inactive_12_mon")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_months_inactive_12_mon_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Months_Inactive_12_mon", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Months_Inactive_12_mon distribution by churn")
save_fig("bivariate_months_inactive_12_mon_box")


Saved: reports/figures/eda_27_bivariate_months_inactive_12_mon_kde.png
Saved: reports/figures/eda_28_bivariate_months_inactive_12_mon_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/1601618064.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_28_bivariate_months_inactive_12_mon_box.png')

In [27]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Contacts_Count_12_mon", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Contacts_Count_12_mon by churn status")
ax.set_xlabel("Contacts_Count_12_mon")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_contacts_count_12_mon_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Contacts_Count_12_mon", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Contacts_Count_12_mon distribution by churn")
save_fig("bivariate_contacts_count_12_mon_box")


Saved: reports/figures/eda_29_bivariate_contacts_count_12_mon_kde.png
Saved: reports/figures/eda_30_bivariate_contacts_count_12_mon_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/486120309.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_30_bivariate_contacts_count_12_mon_box.png')

In [28]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Total_Trans_Amt", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Total_Trans_Amt by churn status")
ax.set_xlabel("Total_Trans_Amt")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_total_trans_amt_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Total_Trans_Amt", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Total_Trans_Amt distribution by churn")
save_fig("bivariate_total_trans_amt_box")


Saved: reports/figures/eda_31_bivariate_total_trans_amt_kde.png
Saved: reports/figures/eda_32_bivariate_total_trans_amt_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/3385635998.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_32_bivariate_total_trans_amt_box.png')

In [29]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Total_Trans_Ct", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Total_Trans_Ct by churn status")
ax.set_xlabel("Total_Trans_Ct")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_total_trans_ct_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Total_Trans_Ct", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Total_Trans_Ct distribution by churn")
save_fig("bivariate_total_trans_ct_box")


Saved: reports/figures/eda_33_bivariate_total_trans_ct_kde.png
Saved: reports/figures/eda_34_bivariate_total_trans_ct_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/198430222.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_34_bivariate_total_trans_ct_box.png')

In [30]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Total_Amt_Chng_Q4_Q1", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Total_Amt_Chng_Q4_Q1 by churn status")
ax.set_xlabel("Total_Amt_Chng_Q4_Q1")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_total_amt_chng_q4_q1_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Total_Amt_Chng_Q4_Q1", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Total_Amt_Chng_Q4_Q1 distribution by churn")
save_fig("bivariate_total_amt_chng_q4_q1_box")


Saved: reports/figures/eda_35_bivariate_total_amt_chng_q4_q1_kde.png
Saved: reports/figures/eda_36_bivariate_total_amt_chng_q4_q1_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/4186988025.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_36_bivariate_total_amt_chng_q4_q1_box.png')

In [31]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Total_Ct_Chng_Q4_Q1", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Total_Ct_Chng_Q4_Q1 by churn status")
ax.set_xlabel("Total_Ct_Chng_Q4_Q1")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_total_ct_chng_q4_q1_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Total_Ct_Chng_Q4_Q1", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Total_Ct_Chng_Q4_Q1 distribution by churn")
save_fig("bivariate_total_ct_chng_q4_q1_box")


Saved: reports/figures/eda_37_bivariate_total_ct_chng_q4_q1_kde.png
Saved: reports/figures/eda_38_bivariate_total_ct_chng_q4_q1_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/506848058.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_38_bivariate_total_ct_chng_q4_q1_box.png')

In [32]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Avg_Utilization_Ratio", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Avg_Utilization_Ratio by churn status")
ax.set_xlabel("Avg_Utilization_Ratio")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_avg_utilization_ratio_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Avg_Utilization_Ratio", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Avg_Utilization_Ratio distribution by churn")
save_fig("bivariate_avg_utilization_ratio_box")


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/787334149.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])


Saved: reports/figures/eda_39_bivariate_avg_utilization_ratio_kde.png


Saved: reports/figures/eda_40_bivariate_avg_utilization_ratio_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_40_bivariate_avg_utilization_ratio_box.png')

In [33]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Credit_Limit", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Credit_Limit by churn status")
ax.set_xlabel("Credit_Limit")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_credit_limit_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Credit_Limit", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Credit_Limit distribution by churn")
save_fig("bivariate_credit_limit_box")


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/4200065477.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])


Saved: reports/figures/eda_41_bivariate_credit_limit_kde.png


Saved: reports/figures/eda_42_bivariate_credit_limit_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_42_bivariate_credit_limit_box.png')

In [34]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=df, x="Months_on_book", hue="is_churn", hue_order=[0, 1],
    palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]], fill=True, alpha=0.4, ax=ax,
)
ax.set_title("Bivariate: Months_on_book by churn status")
ax.set_xlabel("Months_on_book")
handles = [plt.Line2D([0], [0], color=CHURN_PALETTE[k], lw=4, label=CHURN_LABELS[k]) for k in [0, 1]]
ax.legend(handles=handles, title="Churn")
save_fig("bivariate_months_on_book_kde")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=df, x="is_churn", y="Months_on_book", hue="is_churn", palette=[CHURN_PALETTE[0], CHURN_PALETTE[1]],
    legend=False, ax=ax,
)
ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
ax.set_title("Bivariate: Months_on_book distribution by churn")
save_fig("bivariate_months_on_book_box")


Saved: reports/figures/eda_43_bivariate_months_on_book_kde.png
Saved: reports/figures/eda_44_bivariate_months_on_book_box.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1282531/1073369230.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([CHURN_LABELS[0], CHURN_LABELS[1]])
/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_44_bivariate_months_on_book_box.png')

In [35]:
ct = pd.crosstab(df["Gender"], df["is_churn"], normalize="index")[1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ct.plot(kind="barh", color=CHURN_PALETTE[1], ax=ax)
ax.set_title("Bivariate: churn rate by Gender")
ax.set_xlabel("Churn rate (attrited / group)")
ax.set_ylabel("Gender")
save_fig("bivariate_gender_churn_rate")


Saved: reports/figures/eda_45_bivariate_gender_churn_rate.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_45_bivariate_gender_churn_rate.png')

In [36]:
ct = pd.crosstab(df["Education_Level"], df["is_churn"], normalize="index")[1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ct.plot(kind="barh", color=CHURN_PALETTE[1], ax=ax)
ax.set_title("Bivariate: churn rate by Education_Level")
ax.set_xlabel("Churn rate (attrited / group)")
ax.set_ylabel("Education_Level")
save_fig("bivariate_education_level_churn_rate")


Saved: reports/figures/eda_46_bivariate_education_level_churn_rate.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_46_bivariate_education_level_churn_rate.png')

In [37]:
ct = pd.crosstab(df["Marital_Status"], df["is_churn"], normalize="index")[1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ct.plot(kind="barh", color=CHURN_PALETTE[1], ax=ax)
ax.set_title("Bivariate: churn rate by Marital_Status")
ax.set_xlabel("Churn rate (attrited / group)")
ax.set_ylabel("Marital_Status")
save_fig("bivariate_marital_status_churn_rate")


Saved: reports/figures/eda_47_bivariate_marital_status_churn_rate.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_47_bivariate_marital_status_churn_rate.png')

In [38]:
ct = pd.crosstab(df["Income_Category"], df["is_churn"], normalize="index")[1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ct.plot(kind="barh", color=CHURN_PALETTE[1], ax=ax)
ax.set_title("Bivariate: churn rate by Income_Category")
ax.set_xlabel("Churn rate (attrited / group)")
ax.set_ylabel("Income_Category")
save_fig("bivariate_income_category_churn_rate")


Saved: reports/figures/eda_48_bivariate_income_category_churn_rate.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_48_bivariate_income_category_churn_rate.png')

In [39]:
ct = pd.crosstab(df["Card_Category"], df["is_churn"], normalize="index")[1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ct.plot(kind="barh", color=CHURN_PALETTE[1], ax=ax)
ax.set_title("Bivariate: churn rate by Card_Category")
ax.set_xlabel("Churn rate (attrited / group)")
ax.set_ylabel("Card_Category")
save_fig("bivariate_card_category_churn_rate")


Saved: reports/figures/eda_49_bivariate_card_category_churn_rate.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PosixPath('/home/hernan/Escritorio/bank-churners-cursor-bootcamp/reports/figures/eda_49_bivariate_card_category_churn_rate.png')

In [40]:
# Churn rate by category — export table
rows = []
for col in categorical_cols:
    tmp = pd.crosstab(df[col], df["is_churn"], margins=False)
    tmp["churn_rate"] = tmp[1] / tmp.sum(axis=1)
    tmp["feature"] = col
    tmp = tmp.reset_index().rename(columns={col: "category"})
    rows.append(tmp[["feature", "category", 0, 1, "churn_rate"]])

churn_by_cat = pd.concat(rows, ignore_index=True)
churn_by_cat.rename(columns={0: "existing_count", 1: "attrited_count"}, inplace=True)
churn_by_cat.to_csv(TABLES_DIR / "eda_churn_rate_by_category.csv", index=False)
display(churn_by_cat.sort_values("churn_rate", ascending=False).head(12))


is_churn,feature,category,existing_count,attrited_count,churn_rate
21,Card_Category,Platinum,15,5,0.250000
3,Education_Level,Doctorate,356,95,0.210643
20,Card_Category,Gold,95,21,0.181034
6,Education_Level,Post-Graduate,424,92,0.178295
0,Gender,F,4428,930,0.173572
13,Income_Category,$120K +,601,126,0.173315
12,Marital_Status,Unknown,620,129,0.172230
17,Income_Category,Less than $40K,2949,612,0.171862
11,Marital_Status,Single,3275,668,0.169414
8,Education_Level,Unknown,1263,256,0.168532


In [41]:
model_numeric = numeric_cols + ["is_churn"]
corr = df[model_numeric].corr(method="pearson")

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr, cmap="vlag", center=0, annot=False, square=True,
    linewidths=0.5, cbar_kws={"label": "Pearson r"}, ax=ax,
)
ax.set_title("Correlation heatmap (numeric features + is_churn)")
save_fig("correlation_heatmap_pearson")

# Top correlations with churn (exclude self)
churn_corr = (
    corr["is_churn"].drop("is_churn").abs().sort_values(ascending=False)
)
display(churn_corr.head(10).to_frame("abs_corr_with_churn"))


Saved: reports/figures/eda_50_correlation_heatmap_pearson.png


/tmp/ipykernel_1282531/2233523197.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,abs_corr_with_churn
Total_Trans_Ct,0.371403
Total_Ct_Chng_Q4_Q1,0.290054
Total_Revolving_Bal,0.263053
Contacts_Count_12_mon,0.204491
Avg_Utilization_Ratio,0.178410
Total_Trans_Amt,0.168598
Months_Inactive_12_mon,0.152449
Total_Relationship_Count,0.150005
Total_Amt_Chng_Q4_Q1,0.131063
Credit_Limit,0.023873


**Correlation notes (associational):**
- Behavioral and transaction variables often move with `is_churn` in expected directions (e.g., inactivity, lower transaction volume).
- Several balances and limits correlate with each other — expect **multicollinearity**; tree models tolerate this better than linear models.
- Leakage columns are intentionally omitted from this heatmap.


### Ordinal vs nominal encoding (for modeling)

In [42]:
ordinal_mappings = pd.DataFrame([
    {"variable": "Income_Category", "type": "ordinal", "mapping_notes": "1=Less than $40K, 2=$40K-$60K, 3=$60K-$80K, 4=$80K-$120K, 5=$120K+, 6=Unknown"},
    {"variable": "Education_Level", "type": "ordinal", "mapping_notes": "1=Uneducated, 2=High School, 3=College, 4=Graduate, 5=Post-Graduate, 6=Doctorate, 7=Unknown"},
    {"variable": "Card_Category", "type": "ordinal", "mapping_notes": "1=Blue, 2=Silver, 3=Gold, 4=Platinum (product tier proxy)"},
    {"variable": "Gender", "type": "nominal", "mapping_notes": "N-1 dummies (e.g., Female vs reference Male)"},
    {"variable": "Marital_Status", "type": "nominal", "mapping_notes": "N-1 dummies; keep Unknown as explicit category or impute in modeling"},
])
ordinal_mappings.to_csv(TABLES_DIR / "eda_ordinal_mappings.csv", index=False)
display(ordinal_mappings)


,variable,type,mapping_notes
0,Income_Category,ordinal,"1=Less than $40K, 2=$40K-$60K, 3=$60K-$80K, 4=..."
1,Education_Level,ordinal,"1=Uneducated, 2=High School, 3=College, 4=Grad..."
2,Card_Category,ordinal,"1=Blue, 2=Silver, 3=Gold, 4=Platinum (product ..."
3,Gender,nominal,"N-1 dummies (e.g., Female vs reference Male)"
4,Marital_Status,nominal,N-1 dummies; keep Unknown as explicit category...


**Encoding plan**
- **Ordinal:** map to integers starting at 1 following business hierarchy; keep **Unknown** as its own level or impute with a documented rule.
- **Nominal:** use **N-1** dummy encoding to avoid perfect multicollinearity.
- **Target:** use `is_churn` only; drop raw `Attrition_Flag` and leakage columns in training pipelines.


## 5. Key Insights

In [43]:
candidate_features = pd.DataFrame([
    {"group": "behavior", "feature": "Months_Inactive_12_mon", "rationale": "Higher inactivity often aligns with attrition in exploratory views"},
    {"group": "behavior", "feature": "Contacts_Count_12_mon", "rationale": "Contact frequency differs between churned and retained customers"},
    {"group": "transactions", "feature": "Total_Trans_Amt", "rationale": "Lower activity frequently associated with attrited customers"},
    {"group": "transactions", "feature": "Total_Trans_Ct", "rationale": "Complements amount; captures engagement frequency"},
    {"group": "dynamics", "feature": "Total_Amt_Chng_Q4_Q1", "rationale": "Captures recent spending trend"},
    {"group": "dynamics", "feature": "Total_Ct_Chng_Q4_Q1", "rationale": "Captures recent transaction frequency trend"},
    {"group": "utilization", "feature": "Avg_Utilization_Ratio", "rationale": "Credit usage pattern differs by churn status"},
    {"group": "utilization", "feature": "Total_Revolving_Bal", "rationale": "Balance behavior linked to engagement"},
    {"group": "relationship", "feature": "Months_on_book", "rationale": "Tenure differences observed vs churn"},
    {"group": "relationship", "feature": "Total_Relationship_Count", "rationale": "Product depth may relate to stickiness"},
    {"group": "product", "feature": "Card_Category", "rationale": "Product tier mix shifts across churn groups"},
    {"group": "profile", "feature": "Income_Category", "rationale": "Segment-level churn rate variation"},
    {"group": "profile", "feature": "Education_Level", "rationale": "Includes Unknown bucket — document handling"},
])
candidate_features.to_csv(TABLES_DIR / "eda_candidate_features.csv", index=False)
display(candidate_features)


,group,feature,rationale
0,behavior,Months_Inactive_12_mon,Higher inactivity often aligns with attrition ...
1,behavior,Contacts_Count_12_mon,Contact frequency differs between churned and ...
2,transactions,Total_Trans_Amt,Lower activity frequently associated with attr...
3,transactions,Total_Trans_Ct,Complements amount; captures engagement frequency
4,dynamics,Total_Amt_Chng_Q4_Q1,Captures recent spending trend
5,dynamics,Total_Ct_Chng_Q4_Q1,Captures recent transaction frequency trend
6,utilization,Avg_Utilization_Ratio,Credit usage pattern differs by churn status
7,utilization,Total_Revolving_Bal,Balance behavior linked to engagement
8,relationship,Months_on_book,Tenure differences observed vs churn
9,relationship,Total_Relationship_Count,Product depth may relate to stickiness


### Summary

| Question | Finding |
|----------|---------|
| **Churn rate** | ~16.1% attrited (1,627 / 10,127) — imbalanced but workable with stratified modeling |
| **Strongest associations (exploratory)** | Inactivity, contact counts, transaction amount/count and Q4/Q1 change metrics, utilization |
| **Data quality** | No missing cells; literal **Unknown** categories exist; no duplicate IDs |
| **Leakage** | Exclude both `Naive_Bayes_Classifier_*` columns and `CLIENTNUM` from training |
| **Multicollinearity** | Credit limit, open-to-buy, revolving balance, and transactions cluster — choose features deliberately |

### Data quality issues to carry forward
- Literal `"Unknown"` in `Education_Level`, `Marital_Status`, and `Income_Category`
- Some numerics show IQR-flagged extremes — decide cap/winsorize vs trees-only in modeling
- Check utilization values > 1 if present (data definition / rounding)

### Candidate feature engineering
- Ordinal maps for `Income_Category`, `Education_Level`, `Card_Category` (start at 1)
- N-1 dummies for `Gender`, `Marital_Status`
- Document Unknown handling (separate level vs mode imputation)
- Standardize numeric columns for linear models; optional for tree boosting
- Exclude leakage columns and identifier from all training matrices

### Hypotheses (not proven causality)
- Declining transaction trends may precede attrition signals worth monitoring in campaigns
- Higher inactivity and more contacts may coincide with customers at risk (association only)

**Artifacts:** Figures saved under `reports/figures/`; tables under `reports/tables/`. No model artifacts in this EDA phase.
